# Grain Strategy Stepwise Backtest Code

This notebook contains the actual tested code path, split into decision cells.

The intended workflow is:

1. Run one cell.
2. Look at the result.
3. Decide whether the next test is justified.
4. Continue to the next cell.

This is not a static summary notebook. It calls the actual experiment modules used in the research.

## 0. Setup

Run this first. It imports the experiment functions and defines small display helpers.

In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import pandas as pd
try:
    from IPython.display import display, Markdown
except Exception:
    def display(obj):
        print(obj)
    def Markdown(text):
        return text

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 260)

DATA_DIR = "train_set"
COST_ADJUSTED_ONLY = True

from family_regime_model_comparison import run_family_regime_model_comparison
from linear_online_model_experiment import run_linear_online_model_experiment
from soybean_external_signal_experiment import (
    run_soybean_given_only_experiment,
    run_soybean_external_signal_experiment,
)
from soybean_low_vol_switch_experiment import run_soybean_low_vol_switch_experiment
from corn_external_signal_experiment import run_corn_signal_experiment
from corn_abundant_supply_improvement import run_corn_abundant_supply_improvement
from wheat_improvement_experiment import run_wheat_improvement_experiment
from grain_futures_strategy import period_performance


def cost_only(df):
    if COST_ADJUSTED_ONLY and "cost_adjusted" in df.columns:
        return df.loc[df["cost_adjusted"].astype(bool)].copy()
    return df.copy()


def show_cols(df, cols, sort_by=None, ascending=False, n=None):
    out = df.copy()
    cols = [c for c in cols if c in out.columns]
    out = out[cols]
    if sort_by is not None and sort_by in out.columns:
        out = out.sort_values(sort_by, ascending=ascending)
    if n is not None:
        out = out.head(n)
    display(out.reset_index(drop=True))
    return out


def safe_run(label, fn, *args, **kwargs):
    display(Markdown(f"### Running: {label}"))
    try:
        out = fn(*args, **kwargs)
        display(Markdown(f"**Success:** {label}"))
        return out
    except Exception as exc:
        display(Markdown(f"**Failed:** {label}\n\n`{type(exc).__name__}: {exc}`"))
        return None

# 1. Generic Provided-Data Audit

First test: before product-specific work, check whether simple provided-data rules work.

This cell runs:
- average all signals;
- equal family weight;
- IC family selection;
- regime best family;
- regime average families.

In [ ]:
family_audit = safe_run("family/regime provided-data audit", run_family_regime_model_comparison, data_dir=DATA_DIR)
family_results = cost_only(family_audit["results"]) if family_audit is not None else pd.DataFrame()

show_cols(
    family_results,
    [
        "commodity", "strategy", "mode", "cost_adjusted", "oos_sharpe", "oos_pnl", "oos_dd",
        "full_sharpe", "full_dd", "hit_rate", "win_days", "turnover", "overfit_comment",
    ],
    sort_by="commodity",
)

## Decision Check 1

Look for:

- Do simple averages work?
- Does equal family weighting help?
- Does IC selection improve or overfit?
- Do regimes improve enough to justify product-specific regime work?

Expected research read:
- Soybeans improve with regime logic.
- Corn generic tests are weak.
- Wheat directional tests are weak, motivating SRW/HRW pair trading.

# 2. Fitted Model Benchmark: OLS / Kalman / Ridge

Now test whether fitted dynamic weights improve on simple rules.

This is a benchmark, not automatically a final strategy. If fitted models do not beat simple economic rules, they should be rejected because of coefficient-overfit risk.

In [ ]:
linear_audit = safe_run("OLS / Kalman / Ridge benchmark", run_linear_online_model_experiment, data_dir=DATA_DIR)
linear_results = cost_only(linear_audit["results"]) if linear_audit is not None else pd.DataFrame()

show_cols(
    linear_results,
    [
        "commodity", "model", "mode", "cost_adjusted", "test_sharpe", "test_pnl", "test_max_drawdown",
        "full_sharpe", "max_drawdown", "turnover", "avg_gross_exposure",
    ],
    sort_by="commodity",
)

## Decision Check 2

If OLS/Kalman/Ridge do not beat the simple fixed/economic rules, avoid presenting them as final strategies.

Expected research read:
- Corn fitted models remain weak.
- Soybean fitted models are not as good as external/economic blends.
- Wheat directional fitted models do not solve wheat.

# 3. Soybean: Given-Data-Only Tests

Before external data, test the provided soybean signals.

This specifically checks whether Cargill crush and Cargill inventory have value.

In [ ]:
soy_given = safe_run("soybean given-data-only tests", run_soybean_given_only_experiment, data_dir=DATA_DIR)
soy_given_results = cost_only(soy_given["results"]) if soy_given is not None else pd.DataFrame()

show_cols(
    soy_given_results,
    [
        "experiment", "family", "mode", "cost_adjusted", "test_sharpe", "test_pnl", "test_max_drawdown",
        "full_sharpe", "full_pnl", "max_drawdown", "turnover",
    ],
    sort_by="test_sharpe",
    ascending=False,
)

## Decision Check 3

If `given_crush_pressure` is strong, then Cargill crush should be part of the soybean story before adding anything external.

# 4. Soybean: External Economic Signals

Now test whether external soybean economics improve the given-data baseline.

External families include:
- external crush proxy;
- FX/export pressure;
- macro/risk;
- relative grains;
- weather HDD/CDD/GDD.

This cell may need network/data availability for yfinance and Meteostat, depending on your local cache/environment.

In [ ]:
soy_external = safe_run(
    "soybean external signal tests",
    run_soybean_external_signal_experiment,
    data_dir=DATA_DIR,
    include_weather=True,
)
soy_external_results = cost_only(soy_external["results"]) if soy_external is not None else pd.DataFrame()

if soy_external is not None:
    print("External data warnings/errors:", soy_external.get("errors"))

show_cols(
    soy_external_results,
    [
        "experiment", "family", "mode", "cost_adjusted", "test_sharpe", "test_pnl", "test_max_drawdown",
        "full_sharpe", "full_pnl", "max_drawdown", "turnover",
    ],
    sort_by="test_sharpe",
    ascending=False,
    n=20,
)

## Decision Check 4

If `given_crush_plus_weather_hdd_cdd_equal_weight` or `given_physical_external_fundamentals_equal_weight` improves Sharpe/DD, external economics are justified.

If broad `external_equal_family_weight` is weak, do not add every external signal blindly.

# 5. Soybean: Low-Volatility Risk-Control Tests

Performance analysis showed soybean weakness in low-volatility / abundant-supply regimes.

This cell tests fixed, observable risk controls rather than another fitted model.

In [ ]:
soy_low_vol = safe_run("soybean low-volatility switch tests", run_soybean_low_vol_switch_experiment, data_dir=DATA_DIR)
soy_low_vol_results = cost_only(soy_low_vol["results"]) if soy_low_vol is not None else pd.DataFrame()

show_cols(
    soy_low_vol_results,
    [
        "strategy", "cost_adjusted", "oos_sharpe", "oos_pnl", "oos_dd", "full_sharpe", "full_dd",
        "low_vol_sharpe", "low_vol_pnl", "low_vol_dd", "low_price_pnl", "low_price_dd", "turnover",
    ],
    sort_by="oos_sharpe",
    ascending=False,
)

In [ ]:
# Soybean performance-analyst tables.
if soy_low_vol is not None:
    for name, table in soy_low_vol["period_tables"].items():
        display(Markdown(f"### {name}"))
        display(table)

## Decision Check 5

If the low-vol half switch improves DD without introducing fitted coefficients, it is a better fund-style improvement than replacing the low-vol sleeve with a new alpha.

# 6. Corn: Ethanol and External Signals

Corn generic tests were weak, so test corn-specific economic data.

Main economic reason:
- ethanol production/stocks are direct corn-demand signals.

In [ ]:
corn_external = safe_run(
    "corn external / EIA ethanol signal tests",
    run_corn_signal_experiment,
    data_dir=DATA_DIR,
    include_weather=True,
    include_eia=True,
)
corn_external_results = cost_only(corn_external["results"]) if corn_external is not None else pd.DataFrame()

if corn_external is not None:
    print("External data warnings/errors:", corn_external.get("errors"))

show_cols(
    corn_external_results,
    [
        "strategy", "mode", "cost_adjusted", "oos_sharpe", "oos_pnl", "oos_dd",
        "full_sharpe", "full_dd", "turnover",
    ],
    sort_by="oos_sharpe",
    ascending=False,
    n=25,
)

## Decision Check 6

If ethanol has positive OOS but is unstable standalone, use it as a supporting corn signal rather than the entire corn model.

# 7. Corn: Abundant-Supply Risk Guard

Performance analysis showed that corn's major problem was low-price abundant-supply drawdown.

This cell tests no-fit observable guards:
- below long moving average;
- negative medium momentum;
- low/normal volatility;
- curve confirmation.

In [ ]:
corn_guard = safe_run("corn abundant-supply risk-control tests", run_corn_abundant_supply_improvement, data_dir=DATA_DIR)
corn_guard_results = cost_only(corn_guard["results"]) if corn_guard is not None else pd.DataFrame()

show_cols(
    corn_guard_results,
    [
        "strategy", "cost_adjusted", "oos_sharpe", "oos_pnl", "oos_dd", "full_sharpe", "full_dd",
        "low_price_pnl", "low_price_dd", "covid_pnl", "covid_dd", "turnover",
    ],
    sort_by="oos_sharpe",
    ascending=False,
)

In [ ]:
# Corn performance-analyst tables.
if corn_guard is not None:
    for name, table in corn_guard["period_tables"].items():
        display(Markdown(f"### {name}"))
        display(table)

## Decision Check 7

If the abundant-supply guard improves DD materially, corn becomes more usable, but still likely as a smaller satellite sleeve.

# 8. Wheat: SRW / HRW Pair Tests

Directional wheat tests were weak, so test wheat as relative value.

This cell tests:
- SRW/HRW price mean reversion;
- Cargill physical pressure;
- broader physical/curve/COT pair signals;
- cost-control variants;
- trend-aware pair variants.

In [ ]:
wheat_pair = safe_run("wheat SRW/HRW pair tests", run_wheat_improvement_experiment, data_dir=DATA_DIR)
wheat_results = cost_only(wheat_pair["results"]) if wheat_pair is not None else pd.DataFrame()

show_cols(
    wheat_results,
    [
        "strategy", "cost_adjusted", "oos_sharpe", "oos_pnl", "oos_dd", "oos_dd_pct_avg_margin",
        "full_sharpe", "full_dd", "hit_rate", "win_days", "turnover", "formula", "overfit",
    ],
    sort_by="oos_sharpe",
    ascending=False,
    n=25,
)

In [ ]:
# Wheat performance-analyst tables for key variants.
if wheat_pair is not None:
    for key in [
        "pair_price_mr_cargill_90_10_cost_control_cost",
        "pair_price_mr_cargill_80_20_pair_trend_cost_control_cost",
        "pair_price_mr_cargill_trend_conflict_flat_cost_control_cost",
    ]:
        if key in wheat_pair["backtests"]:
            display(Markdown(f"### {key}"))
            display(period_performance(wheat_pair["backtests"][key])[["period", "total_pnl", "sharpe", "max_drawdown", "hit_rate", "days"]])

## Decision Check 8

If SRW/HRW pair trading is much better than standalone wheat, the wheat conclusion should be structural:

> Do not trade SRW and HRW as separate outright alphas. Trade wheat as SRW/HRW relative value if the mandate allows it.

# 9. Final Decision Table

Run this after the earlier cells. It summarizes the research decisions based on the actual outputs you inspected.

In [ ]:
decision_table = pd.DataFrame([
    {
        "Product": "Soybeans",
        "Observed from tests": "Cargill crush worked; external weather/crush/FX improved economics; low-vol was a drawdown problem.",
        "Decision": "Use soybean economic blend with low-vol risk control; keep fitted models in appendix.",
    },
    {
        "Product": "Corn",
        "Observed from tests": "Generic and fitted models were weak; ethanol was relevant; abundant-supply drawdown dominated risk.",
        "Decision": "Use corn only with ethanol/regime context and abundant-supply guard; size as satellite.",
    },
    {
        "Product": "Wheat SRW/HRW",
        "Observed from tests": "Directional wheat failed; SRW/HRW pair MR + Cargill was materially better.",
        "Decision": "Use wheat as relative value, not standalone directional sleeves.",
    },
])
display(decision_table)